# PyTorch Data Loading for MAGIC Telescope Data

### Setup:

1. Need the `magic-gammas.parquet` and `magic-protons.parquet` files.
2. Need a python environment with python 3.10 or higher, pandas, numpy, torch, and matplotlib.
3. Optionally install ipython and ipykernel to use this notebook interactively.

In [1]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from typing import Literal
from pathlib import Path


### Load the data files
- Load parquet files into pandas DataFrames
- Might take minute or two to load into memory

In [ ]:
gamma_file = Path("/path/to/magic-gammas.parquet")
proton_file = Path("/path/to/magic-protons.parquet")

if not gamma_file.exists():
    raise FileNotFoundError(f"File {gamma_file} does not exist")
if not proton_file.exists():
    raise FileNotFoundError(f"File {proton_file} does not exist")

gammas = pd.read_parquet(gamma_file)
protons = pd.read_parquet(proton_file)

print(f"Loaded {len(gammas)} gamma events")
print(f"Loaded {len(protons)} proton events")

Loaded 192794 gamma events
Loaded 105174 proton events


### Create PyTorch Dataset class
- Custom Dataset class that efficiently loads events from pandas DataFrame
- Supports different input types (calibrated images, cleaned images, timing)
- Can combine images and timing as separate channels in the same tensor
- Can include additional image-level features (e.g., Hillas parameters) as a separate feature vector
- Automatically converts numpy arrays to torch tensors
- Always returns a dict with image, particle_id, event_number, run_number, and truth parameters

**Notes on data loading**
- `num_workers=0`: Required for notebooks because classes defined in notebook cells can't be pickled by multiprocessing workers. To enable multiprocessing:
  1. Move `MAGICDataset` class to a separate Python file (e.g., `magic_dataset.py`)
  2. Import it: `from magic_dataset import MAGICDataset`
  3. Then you can set `num_workers > 0` for faster data loading
- `pin_memory=True`: Faster GPU transfer (only use when GPU is available)
- `persistent_workers=True`: Keeps workers alive between epochs (only works when `num_workers > 0`)

In [4]:
class MAGICDataset(Dataset):
    """PyTorch Dataset for MAGIC telescope data.

    Args:
        dataframe: pandas DataFrame containing event data
        particle_type: "gamma" or "hadron" (for labeling)
        telescope_id: Which telescope to use (1 or 2, or "M1"/"M2")
        image_type: "calibrated", "clean", or "timing" - which image data to return
        use_both_telescopes: If True, returns both M1 and M2 images as separate channels
        include_timing: If True, stacks timing data as a separate channel after image data
        additional_features: List of column names to include as additional feature vector
            (e.g., ["hillas_width_m1", "hillas_length_m1", "hillas_size_m1"])
    """

    def __init__(
        self,
        dataframe: pd.DataFrame,
        particle_type: Literal["gamma", "hadron"],
        telescope_id: Literal[1, 2, "M1", "M2"] = 1,
        image_type: Literal["calibrated", "clean", "timing"] = "clean",
        use_both_telescopes: bool = False,
        include_timing: bool = False,
        additional_features: list[str] | None = None,
    ):
        self.dataframe = dataframe.reset_index(drop=True)
        self.particle_type = particle_type
        self.telescope_id = telescope_id
        self.image_type = image_type
        self.use_both_telescopes = use_both_telescopes
        self.include_timing = include_timing
        self.additional_features = additional_features if additional_features else []

        # Convert telescope_id to consistent format
        if telescope_id == 1 or telescope_id == "M1":
            self.telescope_key = "m1"
        else:
            self.telescope_key = "m2"

        # Label: 0 for gamma, 1 for hadron
        self.particle_id = 0 if particle_type == "gamma" else 1
        
        # Validate that include_timing is not used with image_type="timing"
        if include_timing and image_type == "timing":
            raise ValueError("Cannot use include_timing=True with image_type='timing'")
        
        # Validate that additional_features columns exist in dataframe
        if self.additional_features:
            missing_cols = [col for col in self.additional_features if col not in self.dataframe.columns]
            if missing_cols:
                raise ValueError(f"Columns not found in dataframe: {missing_cols}")

    def __len__(self) -> int:
        return len(self.dataframe)

    def __getitem__(self, idx: int) -> dict:
        """Get a single event.

        Returns:
            dict with the following keys:
            - 'image': torch.Tensor with shape [channels, pixels]
            - 'particle_id': torch.Tensor (0 for gamma, 1 for hadron)
            - 'event_number': int
            - 'run_number': int
            - 'truth': dict containing 'particle_id', 'energy', 'theta', 'phi', 'first_interaction_height'
            - 'features': torch.Tensor (only if additional_features is provided)
        """
        row = self.dataframe.iloc[idx]

        # Get image data based on image_type
        if self.image_type == "calibrated":
            if self.use_both_telescopes:
                image_m1 = torch.from_numpy(row["image_m1"]).float()
                image_m2 = torch.from_numpy(row["image_m2"]).float()
                image = torch.stack([image_m1, image_m2], dim=0)  # Shape: [2, 1039]
            else:
                image_key = f"image_{self.telescope_key}"
                image = torch.from_numpy(row[image_key]).float()
        elif self.image_type == "clean":
            if self.use_both_telescopes:
                image_m1 = torch.from_numpy(row["clean_image_m1"]).float()
                image_m2 = torch.from_numpy(row["clean_image_m2"]).float()
                image = torch.stack([image_m1, image_m2], dim=0)  # Shape: [2, 1039]
            else:
                image_key = f"clean_image_{self.telescope_key}"
                image = torch.from_numpy(row[image_key]).float()
        elif self.image_type == "timing":
            if self.use_both_telescopes:
                timing_m1 = torch.from_numpy(row["timing_m1"]).float()
                timing_m2 = torch.from_numpy(row["timing_m2"]).float()
                image = torch.stack([timing_m1, timing_m2], dim=0)  # Shape: [2, 1039]
            else:
                timing_key = f"timing_{self.telescope_key}"
                image = torch.from_numpy(row[timing_key]).float()
        else:
            raise ValueError(f"Unknown image_type: {self.image_type}")

        # If include_timing is True, stack timing data as additional channel(s)
        if self.include_timing:
            if self.use_both_telescopes:
                timing_m1 = torch.from_numpy(row["timing_m1"]).float()
                timing_m2 = torch.from_numpy(row["timing_m2"]).float()
                timing = torch.stack([timing_m1, timing_m2], dim=0)  # Shape: [2, 1039]
                # Stack images first, then timing: [image_m1, image_m2, timing_m1, timing_m2]
                image = torch.cat([image, timing], dim=0)  # Shape: [4, 1039]
            else:
                timing_key = f"timing_{self.telescope_key}"
                timing = torch.from_numpy(row[timing_key]).float()
                # Stack image and timing: [image, timing]
                image = torch.stack([image, timing], dim=0)  # Shape: [2, 1039]
        else:
            # Ensure image is 2D (add channel dimension if single telescope and not including timing)
            if not self.use_both_telescopes:
                image = image.unsqueeze(0)  # Shape: [1, 1039]

        # Extract additional features if requested
        features = None
        if self.additional_features:
            feature_values = [row[col] for col in self.additional_features]
            features = torch.tensor(feature_values, dtype=torch.float32)

        result = {
            "image": image,
            "particle_id": torch.tensor(self.particle_id, dtype=torch.long),
            "event_number": row["event_number"],
            "run_number": row["run_number"],
            "truth": {
                "particle_id": torch.tensor(self.particle_id, dtype=torch.int32),
                "energy": torch.tensor(row["true_energy"], dtype=torch.float32),
                "theta": torch.tensor(row["true_theta"], dtype=torch.float32),
                "phi": torch.tensor(row["true_phi"], dtype=torch.float32),
                "first_interaction_height": torch.tensor(
                    row["true_first_interaction_height"], dtype=torch.float32
                ),
            },
        }
        if features is not None:
            result["features"] = features

        return result


### Create datasets and dataloaders
- Combine gamma and proton datasets
- Create train/validation/test splits
- Set up DataLoaders with proper batching

In [6]:
# Create datasets for gamma and proton events
gamma_dataset = MAGICDataset(gammas, particle_type="gamma", image_type="clean")
proton_dataset = MAGICDataset(protons, particle_type="hadron", image_type="clean")

# If you are doing a training with protons and gammas together: combine datasets using ConcatDataset
from torch.utils.data import ConcatDataset  # noqa: E402

combined_dataset = ConcatDataset([gamma_dataset, proton_dataset])

print(f"Total events: {len(combined_dataset)}")
print(f"  - Gamma events: {len(gamma_dataset)}")
print(f"  - Proton events: {len(proton_dataset)}")

Total events: 297968
  - Gamma events: 192794
  - Proton events: 105174


In [7]:
# Create train/validation/test splits
# Using 70/15/15 split
total_size = len(combined_dataset)
train_size = int(0.7 * total_size)
val_size = int(0.15 * total_size)
test_size = total_size - train_size - val_size

train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(
    combined_dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42),  # For reproducibility
)

print(f"Train set: {len(train_dataset)} events")
print(f"Validation set: {len(val_dataset)} events")
print(f"Test set: {len(test_dataset)} events")

Train set: 208577 events
Validation set: 44695 events
Test set: 44696 events


In [12]:
# Create DataLoaders
batch_size = 4
# Note: num_workers=0 for notebooks because classes defined in notebook cells
# can't be pickled by multiprocessing workers. Set to >0 only if you move
# MAGICDataset to a separate .py file and import it.
num_workers = 0  # Set to 0 for notebooks, or move Dataset class to separate file

# Only use pin_memory if you have a GPU
use_gpu = torch.cuda.is_available()

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=use_gpu,  # Faster GPU transfer (only if GPU available)
    persistent_workers=False,  # Not needed when num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=use_gpu,
    persistent_workers=False,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=use_gpu,
    persistent_workers=False,
)

print("DataLoaders created successfully!")
if num_workers == 0:
    print("  - Using single-threaded data loading (num_workers=0)")
    print("  - To enable multiprocessing, move MAGICDataset to a separate .py file")

DataLoaders created successfully!
  - Using single-threaded data loading (num_workers=0)
  - To enable multiprocessing, move MAGICDataset to a separate .py file


### Test the data loading
- Check batch shapes and types
- Verify particle_id labels are correct
- Note: DataLoader returns batches of dicts, so each key is batched automatically

In [ ]:
# Get a batch from the training loader
batch = next(iter(train_loader))

print("Batch keys:", list(batch.keys()))
print(f"\nImage shape: {batch['image'].shape}")
print("  - Expected: [batch_size, channels, pixels]")
print(f"  - Actual: [{batch['image'].shape[0]}, {batch['image'].shape[1]}, {batch['image'].shape[2]}]")
print(f"\nParticle ID shape: {batch['particle_id'].shape}")
print(f"Particle ID dtype: {batch['particle_id'].dtype}")
print(f"\nSample particle IDs: {batch['particle_id'][:10]}")
print("  - 0 = gamma, 1 = hadron/proton")
print("\nImage batch pixel values stats:")
print(f"  - Min: {batch['image'].min():.2f}")
print(f"  - Max: {batch['image'].max():.2f}")
print(f"  - Mean: {batch['image'].mean():.2f}")
print(f"  - Std: {batch['image'].std():.2f}")

Batch keys: ['image', 'particle_id', 'event_number', 'run_number', 'truth']

Image shape: torch.Size([4, 1, 1183])
  - Expected: [batch_size, channels, pixels]
  - Actual: [4, 1, 1183]

Particle ID shape: torch.Size([4])
Particle ID dtype: torch.int64

Sample particle IDs: tensor([0, 1, 1, 1])
  - 0 = gamma, 1 = hadron/proton

Image batch pixel values stats:
  - Min: 0.00
  - Max: 87.75
  - Mean: 0.74
  - Std: 3.83


### Example: Using both telescopes
- Load data from both M1 and M2 telescopes as separate channels

In [14]:
# Create dataset with both telescopes
gamma_dataset_both = MAGICDataset(
    gammas,
    particle_type="gamma",
    image_type="clean",
    use_both_telescopes=True,
)
proton_dataset_both = MAGICDataset(
    protons,
    particle_type="hadron",
    image_type="clean",
    use_both_telescopes=True,
)

combined_dataset_both = ConcatDataset([gamma_dataset_both, proton_dataset_both])
train_dataset_both, val_dataset_both, test_dataset_both = torch.utils.data.random_split(
    combined_dataset_both,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42),
)

train_loader_both = DataLoader(
    train_dataset_both,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,  # Use 0 for notebooks
    pin_memory=use_gpu,
)

# Check the shape with both telescopes
batch_both = next(iter(train_loader_both))
images_both = batch_both["image"]
particle_ids_both = batch_both["particle_id"]

print(f"Batch shape with both telescopes: {images_both.shape}")
print("  - Expected: [batch_size, 2, pixels]")
print(
    f"  - Actual: [{images_both.shape[0]}, {images_both.shape[1]}, {images_both.shape[2]}]"
)
print("  - Channel 0: M1 telescope")
print("  - Channel 1: M2 telescope")
print(f"\nParticle IDs: {particle_ids_both[:10]}")
print("  - 0 = gamma, 1 = hadron/proton")

Batch shape with both telescopes: torch.Size([4, 2, 1183])
  - Expected: [batch_size, 2, pixels]
  - Actual: [4, 2, 1183]
  - Channel 0: M1 telescope
  - Channel 1: M2 telescope

Particle IDs: tensor([0, 1, 0, 1])
  - 0 = gamma, 1 = hadron/proton


### Example: Using image + timing data
- Load cleaned images AND timing information as separate channels in the same tensor
- Channel 0: cleaned image (photoelectrons)
- Channel 1: timing data (nanoseconds)

In [15]:
# Create dataset with cleaned images AND timing data as separate channels
# Channel 0: cleaned image, Channel 1: timing data
gamma_dataset_timing = MAGICDataset(
    gammas,
    particle_type="gamma",
    image_type="clean",
    telescope_id=1,
    include_timing=True,  # Add timing as a separate channel
)
proton_dataset_timing = MAGICDataset(
    protons,
    particle_type="hadron",
    image_type="clean",
    telescope_id=1,
    include_timing=True,  # Add timing as a separate channel
)

combined_dataset_timing = ConcatDataset([gamma_dataset_timing, proton_dataset_timing])
train_dataset_timing, val_dataset_timing, test_dataset_timing = (
    torch.utils.data.random_split(
        combined_dataset_timing,
        [train_size, val_size, test_size],
        generator=torch.Generator().manual_seed(42),
    )
)

train_loader_timing = DataLoader(
    train_dataset_timing,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,  # Use 0 for notebooks
    pin_memory=use_gpu,
)

# Check the shape with image + timing data
batch_timing = next(iter(train_loader_timing))
images_timing = batch_timing["image"]
particle_ids_timing = batch_timing["particle_id"]

print(f"Batch shape with image + timing: {images_timing.shape}")
print("  - Expected: [batch_size, 2, pixels]")
print("  - Channel 0: cleaned image")
print("  - Channel 1: timing data")
print("\nImage channel (channel 0) stats:")
print(f"  - Min: {images_timing[:, 0, :].min():.2f} p.e.")
print(f"  - Max: {images_timing[:, 0, :].max():.2f} p.e.")
print(f"  - Mean: {images_timing[:, 0, :].mean():.2f} p.e.")
print("\nTiming channel (channel 1) stats:")
print(f"  - Min: {images_timing[:, 1, :].min():.2f} ns")
print(f"  - Max: {images_timing[:, 1, :].max():.2f} ns")
print(f"  - Mean: {images_timing[:, 1, :].mean():.2f} ns")
print(f"  - Std: {images_timing[:, 1, :].std():.2f} ns")
print(f"\nParticle IDs: {particle_ids_timing[:10]}")
print("  - 0 = gamma, 1 = hadron/proton")

Batch shape with image + timing: torch.Size([4, 2, 1183])
  - Expected: [batch_size, 2, pixels]
  - Channel 0: cleaned image
  - Channel 1: timing data

Image channel (channel 0) stats:
  - Min: 0.00 p.e.
  - Max: 187.50 p.e.
  - Mean: 0.85 p.e.

Timing channel (channel 1) stats:
  - Min: -163.50 ns
  - Max: 80.25 ns
  - Mean: 35.72 ns
  - Std: 16.00 ns

Particle IDs: tensor([0, 1, 0, 0])
  - 0 = gamma, 1 = hadron/proton


### Example: Using additional features (Hillas parameters)
- Include image-level parameters like Hillas parameters as a separate feature vector
- Useful for combining image data with reconstructed parameters

In [16]:
# Create dataset with cleaned images AND Hillas parameters as additional features
# The features will be returned as a separate tensor alongside the image
hillas_features = [
    "hillas_width_m1",
    "hillas_length_m1",
    "hillas_size_m1",
    "hillas_width_m2",
    "hillas_length_m2",
    "hillas_size_m2",
]

gamma_dataset_features = MAGICDataset(
    gammas,
    particle_type="gamma",
    image_type="clean",
    telescope_id=1,
    additional_features=hillas_features,
)
proton_dataset_features = MAGICDataset(
    protons,
    particle_type="hadron",
    image_type="clean",
    telescope_id=1,
    additional_features=hillas_features,
)

combined_dataset_features = ConcatDataset([gamma_dataset_features, proton_dataset_features])
train_dataset_features, val_dataset_features, test_dataset_features = (
    torch.utils.data.random_split(
        combined_dataset_features,
        [train_size, val_size, test_size],
        generator=torch.Generator().manual_seed(42),
    )
)

train_loader_features = DataLoader(
    train_dataset_features,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,  # Use 0 for notebooks
    pin_memory=use_gpu,
)

# Check the shape with image + features
batch_features = next(iter(train_loader_features))
images_features = batch_features["image"]
features = batch_features["features"]
particle_ids_features = batch_features["particle_id"]

print(f"Batch shape - images: {images_features.shape}")
print(f"Batch shape - features: {features.shape}")
print(f"Batch shape - particle_ids: {particle_ids_features.shape}")
print(f"\nFeatures included: {hillas_features}")
print("\nFeature stats (first batch):")
for i, feat_name in enumerate(hillas_features):
    print(f"  - {feat_name}: min={features[:, i].min():.2f}, max={features[:, i].max():.2f}, mean={features[:, i].mean():.2f}")
print(f"\nParticle IDs: {particle_ids_features[:10]}")
print("  - 0 = gamma, 1 = hadron/proton")

Batch shape - images: torch.Size([4, 1, 1183])
Batch shape - features: torch.Size([4, 6])
Batch shape - particle_ids: torch.Size([4])

Features included: ['hillas_width_m1', 'hillas_length_m1', 'hillas_size_m1', 'hillas_width_m2', 'hillas_length_m2', 'hillas_size_m2']

Feature stats (first batch):
  - hillas_width_m1: min=23.19, max=55.32, mean=33.13
  - hillas_length_m1: min=45.09, max=182.60, mean=83.70
  - hillas_size_m1: min=184.50, max=1756.82, mean=846.60
  - hillas_width_m2: min=7.42, max=59.20, mean=28.73
  - hillas_length_m2: min=13.98, max=126.16, mean=59.48
  - hillas_size_m2: min=31.05, max=3145.56, mean=974.34

Particle IDs: tensor([1, 1, 1, 0])
  - 0 = gamma, 1 = hadron/proton


### Example: Accessing metadata
- All datasets always return event numbers, run numbers, and truth values
- Truth parameters include particle_id, energy, theta, phi, and first_interaction_height

In [17]:
# Get a single event (all datasets return metadata by default)
sample_event = gamma_dataset[0]

print("Event metadata:")
print(f"  - Event number: {sample_event['event_number']}")
print(f"  - Run number: {sample_event['run_number']}")
print(
    f"  - Particle ID: {sample_event['particle_id'].item()} ({'gamma' if sample_event['particle_id'].item() == 0 else 'hadron'})"
)
print(f"  - Image shape: {sample_event['image'].shape}")
print("\nTruth values:")
for key, value in sample_event["truth"].items():
    if isinstance(value, torch.Tensor):
        print(f"  - {key}: {value.item():.4f}")
    else:
        print(f"  - {key}: {value}")

Event metadata:
  - Event number: 21
  - Run number: 821318
  - Particle ID: 0 (gamma)
  - Image shape: torch.Size([1, 1183])

Truth values:
  - particle_id: 0.0000
  - energy: 341.8620
  - theta: 0.5189
  - phi: 2.7471
  - first_interaction_height: 1279174.5000
